In [1]:
print("Hello World")

Hello World


In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
    "what is encoder and decoder?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
    "The encoder is composed of a stack of N = 6 identical layers. Each layer has twosub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position wise fully connected feed-forward network.  The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack"
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = r"U:\LLMops\data\qa_pairs.csv"
df.to_csv(csv_path, index=False)

In [4]:
df

,question,answer
0,"For customer-facing applications, which compan...","OpenAI models dominate, with 3 of the top 5 an..."
1,What percentage of respondents are using RAG i...,70% of respondents are using RAG in some form.
2,How often are most respondents updating their ...,More than 50% update their models at least mon...
3,what is encoder and decoder?,The encoder is composed of a stack of N = 6 id...


In [5]:
from langsmith import Client

client = Client()
dataset_name = "MultiDocChat"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for MultiDocChat",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['5c57dfef-441f-4da5-80b8-14f6a8452055',
  '126f9ccf-c7ab-4a49-a8b8-7a11fdd26874',
  'a6fa5bd3-1510-4607-84a6-5bcb77af181b',
  '94200d29-fba6-4ef4-9365-56cec6d26aeb'],
 'count': 4,
 'as_of': '2026-02-24T17:17:01.652491552Z'}

In [6]:
import sys
sys.path.append("U:\LLMops")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = r"U:\LLMops\data\attention_all_you_need.pdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [7]:
import sys
print(sys.executable)

u:\LLMops\venv\python.exe


In [8]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-02-24T17:17:08.430896Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:08.431896Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:17:08.432895Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-02-24T17:17:08.437357Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260224_224708_7302cc65", "temp_dir": "data\\session_20260224_224708_7302cc65", "faiss_dir": "faiss_index\\session_20260224_224708_7302cc65", "sessionized": true, "timestamp": "2026-02-24T17:17:08.438363Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "attention_all_you_need.pdf", "saved_as": "data\\session_20260224_224708_7302cc65\\00d0c458.pdf", "timestamp": "2026-02-24T17:17:08.441365Z", "level": "info", "event": "File 

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: I don't know.


In [9]:
# from langsmith.evaluation import evaluate, LangChainStringEvaluator

In [10]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-02-24T17:17:17.866469Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:17.867650Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:17:17.868657Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-02-24T17:17:17.872657Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260224_224717_d7f2dd8e", "temp_dir": "data\\session_20260224_224717_d7f2dd8e", "faiss_dir": "faiss_index\\session_20260224_224717_d7f2dd8e", "sessionized": true, "timestamp": "2026-02-24T17:17:17.874658Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "attention_all_you_need.pdf", "saved_as": "data\\session_20260224_224717_d7f2dd8e\\816db082.pdf", "timestamp": "2026-02-24T17:17:17.877688Z", "level": "info", "event": "File 

Testing all questions from the dataset:



{"count": 15, "timestamp": "2026-02-24T17:17:19.035223Z", "level": "info", "event": "Documents loaded"}
{"chunks": 52, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-02-24T17:17:19.038255Z", "level": "info", "event": "Documents split"}
{"model": "nomic-ai/nomic-embed-text-v1.5", "timestamp": "2026-02-24T17:17:19.039214Z", "level": "info", "event": "Loading embedding model"}
{"added": 1, "index": "faiss_index\\session_20260224_224717_d7f2dd8e", "timestamp": "2026-02-24T17:17:21.531067Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-02-24T17:17:21.531067Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-02-24T17:17:21.531067Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:21.531067Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: I don't know.

--------------------------------------------------------------------------------



{"count": 15, "timestamp": "2026-02-24T17:17:24.022218Z", "level": "info", "event": "Documents loaded"}
{"chunks": 52, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-02-24T17:17:24.022218Z", "level": "info", "event": "Documents split"}
{"model": "nomic-ai/nomic-embed-text-v1.5", "timestamp": "2026-02-24T17:17:24.028933Z", "level": "info", "event": "Loading embedding model"}
{"added": 1, "index": "faiss_index\\session_20260224_224722_e3c2fe13", "timestamp": "2026-02-24T17:17:26.542313Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-02-24T17:17:26.542313Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-02-24T17:17:26.542313Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:26.542313Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:

Q2: What percentage of respondents are using RAG in some form?
A2: I don't know.

--------------------------------------------------------------------------------



{"count": 15, "timestamp": "2026-02-24T17:17:30.240269Z", "level": "info", "event": "Documents loaded"}
{"chunks": 52, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-02-24T17:17:30.240269Z", "level": "info", "event": "Documents split"}
{"model": "nomic-ai/nomic-embed-text-v1.5", "timestamp": "2026-02-24T17:17:30.240269Z", "level": "info", "event": "Loading embedding model"}
{"added": 1, "index": "faiss_index\\session_20260224_224729_2f826a06", "timestamp": "2026-02-24T17:17:32.948211Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-02-24T17:17:32.948211Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-02-24T17:17:32.948211Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:32.948211Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:

Q3: How often are most respondents updating their models?
A3: I don't know.

--------------------------------------------------------------------------------



{"count": 15, "timestamp": "2026-02-24T17:17:35.674133Z", "level": "info", "event": "Documents loaded"}
{"chunks": 52, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-02-24T17:17:35.674133Z", "level": "info", "event": "Documents split"}
{"model": "nomic-ai/nomic-embed-text-v1.5", "timestamp": "2026-02-24T17:17:35.674133Z", "level": "info", "event": "Loading embedding model"}
{"added": 1, "index": "faiss_index\\session_20260224_224734_28cf3df9", "timestamp": "2026-02-24T17:17:39.205064Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-02-24T17:17:39.205064Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-02-24T17:17:39.205064Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:39.205064Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:

Q4: what is encoder and decoder?
A4: The encoder transforms an input sequence of symbols (x₁,…,xₙ) into a sequence of continuous representations z = (z₁,…,zₙ) using stacked self‑attention and feed‑forward layers. The decoder takes those encoded representations and, in an auto‑regressive manner, generates the output symbols (y₁,…,yₘ) one at a time, attending both to previously generated tokens and to the encoder’s output.

--------------------------------------------------------------------------------



In [11]:
# !pip install langchain_community

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from langsmith.evaluation import evaluate

### Evaluation Code 1:

In [14]:
experiment_results = evaluate(
    answer_ai_report_question,
    data="MultiDocChat",
    # evaluators=["qa"],
    experiment_prefix="test-agenticAIReport-qa-rag",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

u:\LLMops\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-f8b8599e' at:
https://smith.langchain.com/o/9a0c623e-ce2d-56ad-9784-8f38707079e0/datasets/5f4591c9-c813-4efd-b4b0-b8b5413bb7e6/compare?selectedSessions=9b6ab4e7-5434-437c-b48b-118d24d24a6e




0it [00:00, ?it/s]{"timestamp": "2026-02-24T17:17:49.101375Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-02-24T17:17:49.101375Z", "level": "info", "event": "Loaded API_KEYS from ECS secret"}
{"keys": {"GROQ_API_KEY": "gsk_ph...", "FIREWORKS_API_KEY": "fw_3ZY..."}, "timestamp": "2026-02-24T17:17:49.109512Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-02-24T17:17:49.109512Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260224_224749_d4039c77", "temp_dir": "data\\session_20260224_224749_d4039c77", "faiss_dir": "faiss_index\\session_20260224_224749_d4039c77", "sessionized": true, "timestamp": "2026-02-24T17:17:49.115852Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "attention_all_you_need.pdf", "saved_as": "data\\session_20260224_224749_d4039c77\\79527f81.pdf", "timestamp": "2026-02-24T17:17:49.120158Z", "level": "info

In [15]:
experiment_results

,inputs.question,outputs.answer,error,reference.answer,execution_time,example_id,id
0,What percentage of respondents are using RAG i...,I don't know.,None,70% of respondents are using RAG in some form.,5.048973,126f9ccf-c7ab-4a49-a8b8-7a11fdd26874,019c90a8-36ad-7f61-ba0c-e2213316f622
1,"For customer-facing applications, which compan...",I don't know.,None,"OpenAI models dominate, with 3 of the top 5 an...",6.647040,5c57dfef-441f-4da5-80b8-14f6a8452055,019c90a8-4a66-71d1-9107-941a6ab129ea
2,what is encoder and decoder?,The encoder transforms an input sequence of sy...,None,The encoder is composed of a stack of N = 6 id...,5.322797,94200d29-fba6-4ef4-9365-56cec6d26aeb,019c90a8-645d-7ad2-8ad5-99577e39d6ff
3,How often are most respondents updating their ...,I don't know.,None,More than 50% update their models at least mon...,6.232604,a6fa5bd3-1510-4607-84a6-5bcb77af181b,019c90a8-7929-76a1-bba8-c22d7797876e


In [ ]:
# Evaluation Code 2:

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate  # Only this import needed

client = Client()

# Custom evaluator function (replaces LangGraphStringEvaluator)
def string_evaluator(run, example):
    # Compare prediction output to reference; customize logic as needed
    prediction = run.outputs.get("output", "")
    reference = example.outputs.get("output", "")
    score = 1.0 if prediction.strip() == reference.strip() else 0.0
    return {"key": "string_match", "score": score}

# Run eval on your dataset (no special LangGraph evaluator needed)
results = evaluate(
    answer_ai_report_question,  # Your @traceable LangGraph app
    data="AgenticAIReportGoldens",  # Or dataset ID; filters applied in UI/experiment view
    evaluators=[string_evaluator],
    experiment_prefix="langgraph-eval-feb2026",
    client=client,
    max_concurrency=4  # Parallelizes for speed
)


In [ ]:
results

## Custom Correctness Evaluator

Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignment


In [ ]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

### Run Evaluation with Custom Correctness Evaluator


In [ ]:
from langsmith.evaluation import evaluate

In [ ]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")


### Optional: Combine Multiple Evaluators

You can use multiple evaluators together to get different perspectives on your RAG system's performance.


In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )
